# Phase 2, Workshop 04 — Wiring Up Game Logic

You have a beautiful visual board and you have working game logic from Phase 1. Now it is time to connect them. In this workshop you will take the core game functions (rolling dice, handling spaces, buying properties, jail) and wire them into the visual board, so that every game action updates both the board display and the sidebar.

**What you will learn:**
- Integrating Phase 1 game logic into the Phase 2 visual layout
- Updating the `updateAll` function to refresh both board and sidebar
- Showing dice results in the board centre
- Displaying contextual information as the game progresses

**Time:** roughly 15 to 20 minutes.

## Section 1 — The updateAll Function

In Phase 1, `updateUI` refreshed the sidebar (cash, position, jail status, buttons). In Phase 2, `updateAll` does that *and also*:

1. Updates the **turn badge** at the top of the sidebar
2. Places **tokens** on the correct board cells
3. Shows **ownership indicators** on properties
4. Enables/disables the correct **buttons**
5. Updates **player stats** (cash, position, jail)

The key insight is that this single function is called after *every* game action. It reads the state and rebuilds the entire display. This means you never have to worry about the display being out of sync with the data.

## Section 2 — Compact Board Data

Phase 2 uses shorter property names in the board data to fit inside the small cells. Compare:

| Phase 1 | Phase 2 |
|---|---|
| `name: "Old Kent Road"` | `n: "Old Kent Rd"` |
| `type: "property"` | `t: "prop"` |
| `cost: 60` | `c: 60` |
| `rent: 10` | `r: 10` |

The shorter keys save space and make the large board array easier to read. The game logic works exactly the same way, just with different property names.

## Section 3 — The Integrated Game

This is a big cell. It combines the visual board from Workshops 01 to 03 with the complete game logic from Phase 1. Read through it carefully; the structure should feel familiar from both phases.

In [ ]:
import os

project_folder = os.path.join(os.path.expanduser("~"), "text_monopoly", "phase2")
os.makedirs(project_folder, exist_ok=True)

html_content = """<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Phase 2 - Game Logic Wired Up</title>
    <style>
        @import url('https://fonts.googleapis.com/css2?family=Outfit:wght@400;600;700;900&family=JetBrains+Mono:wght@400;700&display=swap');
        :root {
            --bg: #1a1520; --board-bg: #f5f0e8; --board-border: #1a1520;
            --board-center: #dcd4c8; --text-dark: #1a1520; --text-light: #f5f0e8;
            --surface: #231e2a; --border: #3a3344;
            --p1: #e84855; --p2: #3185fc; --gold: #f2a93b; --green: #4ecb71;
            --danger: #e84855; --dim: #8a8394;
            --brown: #8B4513; --sky: #87CEEB; --pink: #d63384; --orange: #fd7e14;
            --red: #e84855; --yellow: #ffc107; --dgreen: #198754; --navy: #0d6efd;
            --station: #444; --utility: #aaa;
        }
        * { margin: 0; padding: 0; box-sizing: border-box; }
        body { font-family: 'Outfit', sans-serif; background: var(--bg); color: var(--text-light); min-height: 100vh; display: flex; flex-direction: column; align-items: center; padding: 15px; }
        .header { text-align: center; margin-bottom: 12px; }
        .header h1 { font-size: 1.4rem; font-weight: 900; color: var(--gold); }
        .header .sub { color: var(--dim); font-size: 0.75rem; }
        .game-layout { display: flex; gap: 16px; align-items: flex-start; max-width: 1200px; width: 100%; }

        .board { display: grid; grid-template-columns: 72px repeat(9, 56px) 72px; grid-template-rows: 72px repeat(9, 56px) 72px; gap: 1px; background: var(--board-border); border: 3px solid var(--board-border); border-radius: 8px; overflow: hidden; font-size: 0.5rem; flex-shrink: 0; }
        .cell { background: var(--board-bg); color: var(--text-dark); display: flex; flex-direction: column; align-items: center; justify-content: center; position: relative; padding: 2px; text-align: center; line-height: 1.15; font-weight: 600; overflow: hidden; }
        .cell .color-bar { position: absolute; width: 100%; height: 8px; left: 0; }
        .side-top .color-bar { bottom: 0; } .side-bottom .color-bar { top: 0; }
        .side-left .color-bar { top: 0; left: unset; right: 0; width: 8px; height: 100%; }
        .side-right .color-bar { top: 0; left: 0; width: 8px; height: 100%; }
        .cell .name { font-size: 0.45rem; font-weight: 700; z-index: 1; } .cell .price { font-size: 0.4rem; color: #666; z-index: 1; } .cell .icon { font-size: 0.9rem; z-index: 1; }
        .corner { font-size: 0.55rem; font-weight: 900; }
        .tokens { display: flex; gap: 2px; z-index: 2; margin-top: 2px; }
        .token { width: 14px; height: 14px; border-radius: 50%; display: flex; align-items: center; justify-content: center; font-size: 0.5rem; font-weight: 900; color: #fff; box-shadow: 0 1px 3px rgba(0,0,0,0.4); }
        .token-p1 { background: var(--p1); } .token-p2 { background: var(--p2); }
        .board-center { grid-column: 2/11; grid-row: 2/11; background: var(--board-center); display: flex; flex-direction: column; align-items: center; justify-content: center; gap: 6px; }
        .board-center .title { font-size: 1.8rem; font-weight: 900; color: var(--text-dark); letter-spacing: -1px; }
        .board-center .dice-area { display: flex; gap: 10px; font-size: 2.2rem; }
        .board-center .info { font-size: 0.65rem; color: #777; text-align: center; max-width: 200px; }

        .sidebar { flex: 1; min-width: 260px; display: flex; flex-direction: column; gap: 10px; }
        .panel { background: var(--surface); border: 1px solid var(--border); border-radius: 10px; padding: 14px; }
        .panel h3 { font-size: 0.85rem; font-weight: 700; margin-bottom: 8px; }
        .turn-badge { text-align: center; padding: 10px; border-radius: 10px; font-weight: 700; font-size: 0.9rem; }
        .turn-p1 { background: rgba(232,72,85,0.2); color: var(--p1); border: 1px solid rgba(232,72,85,0.3); }
        .turn-p2 { background: rgba(49,133,252,0.2); color: var(--p2); border: 1px solid rgba(49,133,252,0.3); }
        .stat { display: flex; justify-content: space-between; padding: 3px 0; font-size: 0.75rem; }
        .stat .label { color: var(--dim); } .stat .val { font-weight: 700; font-family: 'JetBrains Mono', monospace; }
        .player-info { display: flex; align-items: center; gap: 8px; margin-bottom: 6px; }
        .player-dot { width: 12px; height: 12px; border-radius: 50%; }

        button { font-family: 'Outfit', sans-serif; width: 100%; padding: 11px; border-radius: 8px; border: 1px solid var(--border); background: var(--bg); color: var(--text-light); font-weight: 700; font-size: 0.85rem; cursor: pointer; margin-bottom: 6px; transition: all 0.2s; }
        button:hover:not(:disabled) { border-color: var(--gold); background: rgba(242,169,59,0.1); }
        button:disabled { opacity: 0.25; cursor: not-allowed; }
        button.primary { background: linear-gradient(135deg, var(--p1), var(--gold)); border: none; color: #fff; font-size: 0.95rem; }
        button.primary:hover:not(:disabled) { filter: brightness(1.1); }

        .log-box { height: 160px; overflow-y: auto; font-family: 'JetBrains Mono', monospace; font-size: 0.65rem; line-height: 1.6; }
        .log-box::-webkit-scrollbar { width: 4px; } .log-box::-webkit-scrollbar-thumb { background: var(--border); border-radius: 2px; }
        .log-line.sys { color: var(--dim); } .log-line.p1 { color: var(--p1); } .log-line.p2 { color: var(--p2); } .log-line.ev { color: var(--gold); } .log-line.bad { color: var(--danger); } .log-line.good { color: var(--green); }
    </style>
</head>
<body>
    <div class="header"><h1>🎩 MONOPOLY — Phase 2</h1><div class="sub">Visual board with full game logic</div></div>
    <div class="game-layout">
        <div class="board-wrapper"><div class="board" id="board"></div></div>
        <div class="sidebar">
            <div class="turn-badge turn-p1" id="turnBadge">🎲 Player 1's Turn</div>
            <div class="panel"><div class="player-info"><div class="player-dot" style="background:var(--p1)"></div><h3>Player 1</h3></div><div class="stat"><span class="label">Cash</span><span class="val" id="s_p1cash">£1500</span></div><div class="stat"><span class="label">Position</span><span class="val" id="s_p1pos">GO</span></div><div class="stat"><span class="label">Jail</span><span class="val" id="s_p1jail">No</span></div></div>
            <div class="panel"><div class="player-info"><div class="player-dot" style="background:var(--p2)"></div><h3>Player 2</h3></div><div class="stat"><span class="label">Cash</span><span class="val" id="s_p2cash">£1500</span></div><div class="stat"><span class="label">Position</span><span class="val" id="s_p2pos">GO</span></div><div class="stat"><span class="label">Jail</span><span class="val" id="s_p2jail">No</span></div></div>
            <div class="panel"><h3>Actions</h3><button class="primary" id="rollBtn" onclick="rollDice()">🎲 Roll Dice</button><button id="buyBtn" onclick="buyProperty()" disabled>💰 Buy Property</button><button id="endBtn" onclick="endTurn()" disabled>➡️ End Turn</button></div>
            <div class="panel"><h3>Game Log</h3><div class="log-box" id="logBox"></div></div>
        </div>
    </div>
    <script>
        const B=[{n:"GO",t:"go",icon:"🏁"},{n:"Old Kent Rd",t:"prop",c:60,r:10,col:"var(--brown)"},{n:"Chest",t:"chest",icon:"📦"},{n:"Whitechapel",t:"prop",c:60,r:10,col:"var(--brown)"},{n:"Income Tax",t:"tax",a:200,icon:"💸"},{n:"Kings Cross",t:"prop",c:200,r:50,col:"var(--station)"},{n:"Angel",t:"prop",c:100,r:20,col:"var(--sky)"},{n:"Chance",t:"chance",icon:"❓"},{n:"Euston Rd",t:"prop",c:100,r:20,col:"var(--sky)"},{n:"Pentonville",t:"prop",c:120,r:25,col:"var(--sky)"},{n:"Visiting",t:"visit",icon:"👀"},{n:"Pall Mall",t:"prop",c:140,r:30,col:"var(--pink)"},{n:"Electric Co",t:"prop",c:150,r:35,col:"var(--utility)"},{n:"Whitehall",t:"prop",c:140,r:30,col:"var(--pink)"},{n:"Northumber.",t:"prop",c:160,r:35,col:"var(--pink)"},{n:"Marylebone",t:"prop",c:200,r:50,col:"var(--station)"},{n:"Bow St",t:"prop",c:180,r:40,col:"var(--orange)"},{n:"Chest",t:"chest",icon:"📦"},{n:"Marlborough",t:"prop",c:180,r:40,col:"var(--orange)"},{n:"Vine St",t:"prop",c:200,r:45,col:"var(--orange)"},{n:"Free Parking",t:"free",icon:"🅿️"},{n:"Strand",t:"prop",c:220,r:50,col:"var(--red)"},{n:"Chance",t:"chance",icon:"❓"},{n:"Fleet St",t:"prop",c:220,r:50,col:"var(--red)"},{n:"Trafalgar Sq",t:"prop",c:240,r:55,col:"var(--red)"},{n:"Fenchurch",t:"prop",c:200,r:50,col:"var(--station)"},{n:"Leicester Sq",t:"prop",c:260,r:60,col:"var(--yellow)"},{n:"Coventry St",t:"prop",c:260,r:60,col:"var(--yellow)"},{n:"Water Works",t:"prop",c:150,r:35,col:"var(--utility)"},{n:"Piccadilly",t:"prop",c:280,r:65,col:"var(--yellow)"},{n:"GO TO JAIL",t:"jail",icon:"🚔"},{n:"Regent St",t:"prop",c:300,r:70,col:"var(--dgreen)"},{n:"Oxford St",t:"prop",c:300,r:70,col:"var(--dgreen)"},{n:"Chest",t:"chest",icon:"📦"},{n:"Bond St",t:"prop",c:320,r:75,col:"var(--dgreen)"},{n:"Liverpool St",t:"prop",c:200,r:50,col:"var(--station)"},{n:"Chance",t:"chance",icon:"❓"},{n:"Park Lane",t:"prop",c:350,r:85,col:"var(--navy)"},{n:"Super Tax",t:"tax",a:100,icon:"💸"},{n:"Mayfair",t:"prop",c:400,r:100,col:"var(--navy)"}];
        const CHANCE=[{text:"Advance to GO! +£200",fn:p=>{p.pos=0;p.cash+=200;}},{text:"Bank dividend: +£50",fn:p=>{p.cash+=50;}},{text:"Go back 3 spaces",fn:p=>{p.pos=(p.pos-3+40)%40;}},{text:"Go to Jail!",fn:p=>{p.pos=10;p.jail=true;p.jt=0;}},{text:"Repairs: -£150",fn:p=>{p.cash-=150;}},{text:"Crossword prize: +£100",fn:p=>{p.cash+=100;}},{text:"Speeding fine: -£50",fn:p=>{p.cash-=50;}}];
        const CHEST=[{text:"Bank error: +£200",fn:p=>{p.cash+=200;}},{text:"Doctor fees: -£50",fn:p=>{p.cash-=50;}},{text:"Stock sale: +£45",fn:p=>{p.cash+=45;}},{text:"Holiday fund: +£100",fn:p=>{p.cash+=100;}},{text:"School fees: -£150",fn:p=>{p.cash-=150;}},{text:"Tax refund: +£20",fn:p=>{p.cash+=20;}},{text:"Go to Jail!",fn:p=>{p.pos=10;p.jail=true;p.jt=0;}}];

        function boardToGrid(i){if(i<=10)return{row:10,col:10-i};if(i<=20)return{row:10-(i-10),col:0};if(i<=30)return{row:0,col:i-20};return{row:i-30,col:10};}
        function getSide(i){if(i<=10)return"side-bottom";if(i<=20)return"side-left";if(i<=30)return"side-top";return"side-right";}

        // Build board
        function buildBoard(){const board=document.getElementById("board");board.innerHTML="";for(let i=0;i<40;i++){const g=boardToGrid(i),sp=B[i],side=getSide(i),isCorner=[0,10,20,30].includes(i);const div=document.createElement("div");div.className="cell "+side+(isCorner?" corner":"");div.style.gridRow=g.row+1;div.style.gridColumn=g.col+1;div.id="cell-"+i;let h="";if(sp.t==="prop"&&sp.col)h+='<div class="color-bar" style="background:'+sp.col+'"></div>';if(sp.icon)h+='<div class="icon">'+sp.icon+'</div>';h+='<div class="name">'+sp.n+'</div>';if(sp.c)h+='<div class="price">£'+sp.c+'</div>';h+='<div class="tokens" id="tokens-'+i+'"></div>';div.innerHTML=h;board.appendChild(div);}const center=document.createElement("div");center.className="board-center";center.innerHTML='<div class="title">MONOPOLY</div><div class="dice-area" id="diceArea">🎲 🎲</div><div class="info" id="centerInfo">Roll the dice to start!</div>';board.appendChild(center);}

        // Game state
        let G={players:[{name:"Player 1",cash:1500,pos:0,props:[],jail:false,jt:0},{name:"Player 2",cash:1500,pos:0,props:[],jail:false,jt:0}],cur:0,phase:"roll",over:false};

        function log(txt,cls){const box=document.getElementById("logBox");const d=document.createElement("div");d.className="log-line "+(cls||"sys");d.textContent="› "+txt;box.appendChild(d);box.scrollTop=box.scrollHeight;}
        function owner(pos){for(let i=0;i<G.players.length;i++)if(G.players[i].props.includes(pos))return i;return null;}
        function checkBust(p){if(p.cash<=0){G.over=true;const w=G.players.find(x=>x!==p);log(p.name+" is BANKRUPT!","bad");log(w.name+" WINS! 🏆","good");}}

        function handleLanding(p,sp,cls){switch(sp.t){case"go":p.cash+=200;log("+£200 from GO!","good");G.phase="end";break;case"prop":const ow=owner(p.pos);if(ow===null){if(p.cash>=sp.c){G.phase="buy";log("Available for £"+sp.c+"!","ev");}else{log("Cannot afford £"+sp.c,"sys");G.phase="end";}}else if(ow!==G.cur){p.cash-=sp.r;G.players[ow].cash+=sp.r;log("Rent £"+sp.r+" to "+G.players[ow].name,"bad");G.phase="end";checkBust(p);}else{log("Own this!","sys");G.phase="end";}break;case"tax":p.cash-=sp.a;log("Tax: -£"+sp.a,"bad");G.phase="end";checkBust(p);break;case"jail":p.pos=10;p.jail=true;p.jt=0;log("GO TO JAIL! 🚔","bad");G.phase="end";break;case"chance":const ch=CHANCE[Math.floor(Math.random()*CHANCE.length)];log("Chance: "+ch.text,"ev");ch.fn(p);G.phase="end";checkBust(p);break;case"chest":const cc=CHEST[Math.floor(Math.random()*CHEST.length)];log("Chest: "+cc.text,"ev");cc.fn(p);G.phase="end";checkBust(p);break;default:G.phase="end";break;}}

        function rollDice(){if(G.phase!=="roll"||G.over)return;const d1=Math.floor(Math.random()*6)+1,d2=Math.floor(Math.random()*6)+1,total=d1+d2;const de=["","⚀","⚁","⚂","⚃","⚄","⚅"];document.getElementById("diceArea").textContent=de[d1]+" "+de[d2];const p=G.players[G.cur],cls=G.cur===0?"p1":"p2";log(p.name+" rolled "+d1+"+"+d2+" = "+total,cls);if(p.jail){p.jt++;if(d1===d2){p.jail=false;log("Doubles! Escaped jail!","ev");}else if(p.jt>=3){p.jail=false;p.cash-=50;log("Paid £50 jail fine","bad");}else{log("Still in jail ("+p.jt+"/3)","bad");G.phase="end";updateAll();return;}}const old=p.pos;p.pos=(p.pos+total)%40;if(p.pos<old){p.cash+=200;log("Passed GO! +£200","good");}const sp=B[p.pos];log("Landed on "+sp.n,cls);document.getElementById("centerInfo").textContent=p.name+" → "+sp.n;handleLanding(p,sp,cls);updateAll();}
        function buyProperty(){if(G.phase!=="buy")return;const p=G.players[G.cur],sp=B[p.pos];p.cash-=sp.c;p.props.push(p.pos);log("Bought "+sp.n+" for £"+sp.c+"! 🏠","good");G.phase="end";updateAll();}
        function endTurn(){if(G.phase!=="end"&&G.phase!=="buy")return;G.cur=(G.cur+1)%2;G.phase="roll";updateAll();}

        function updateAll(){const tb=document.getElementById("turnBadge");tb.textContent="🎲 "+G.players[G.cur].name+"'s Turn";tb.className="turn-badge turn-p"+(G.cur+1);for(let i=0;i<2;i++){const pl=G.players[i];document.getElementById("s_p"+(i+1)+"cash").textContent="£"+pl.cash;document.getElementById("s_p"+(i+1)+"pos").textContent=B[pl.pos].n;document.getElementById("s_p"+(i+1)+"jail").textContent=pl.jail?"Yes ("+pl.jt+"/3)":"No";}for(let i=0;i<40;i++){const tc=document.getElementById("tokens-"+i);if(tc)tc.innerHTML="";}for(let i=0;i<2;i++){const tc=document.getElementById("tokens-"+G.players[i].pos);if(tc){const tok=document.createElement("div");tok.className="token token-p"+(i+1);tok.textContent=i+1;tc.appendChild(tok);}}document.getElementById("rollBtn").disabled=G.phase!=="roll"||G.over;document.getElementById("buyBtn").disabled=G.phase!=="buy"||G.over;document.getElementById("endBtn").disabled=(G.phase!=="end"&&G.phase!=="buy")||G.over;}

        buildBoard();log("Welcome to Visual Monopoly!","sys");log("Roll the dice to begin!","sys");updateAll();
    </script>
</body>
</html>
"""

filepath = os.path.join(project_folder, "p2_04_game_logic.html")
with open(filepath, "w", encoding="utf-8") as f:
    f.write(html_content)

print(f"Game with visual board saved to: {filepath}")
print("This is a fully playable game with the visual board!")

## Section 4 — What Changed from Phase 1?

The game logic is almost identical. The main differences are:

1. **Property names** use shorter keys (`n`, `t`, `c`, `r`, `col` instead of `name`, `type`, `cost`, `rent`, `color`).
2. **Player positions** use `pos` instead of `position`, `jail` instead of `inJail`, `jt` instead of `jailTurns`, `props` instead of `properties`.
3. **The `updateAll` function** does more work: it updates the sidebar *and* redraws tokens on the board.
4. **Dice display** appears in the board centre instead of the sidebar.
5. **The game log** moves to a compact sidebar panel with colour-coded messages.

The logic itself (rolling, movement, buying, rent, jail, bankruptcy) is unchanged.

## Section 5 — Challenge

1. The game currently does not show ownership indicators on the board. We will add those in Workshop 05, but try adding them yourself first (hint: loop through all 40 cells in `updateAll`, check `owner(i)`, and add a small coloured dot if owned).
2. Add the dice emoji to the `log` output alongside the text.
3. Test the jail system thoroughly: get a player sent to jail and verify they can only escape by rolling doubles or serving three turns.

In **Workshop 05** you will add ownership indicators and property tags. 🏠